# Build a LanceDB VDB workflow with NeMo Retriever

This tutorial ingests extraction results into a local **LanceDB** table using the **NeMo Retriever Library** graph pipeline and queries it with `Retriever`.

**Important:** For production deployments, use the [NeMo Retriever Helm chart](https://github.com/NVIDIA/NeMo-Retriever/tree/main/nemo_retriever/helm). Docker Compose in this repository is unsupported developer-only tooling.

## Prerequisites

- [NeMo-Retriever](https://github.com/NVIDIA/NeMo-Retriever) checkout with `pip install "nemo-retriever[local]"` (see [nemo_retriever/README.md](https://github.com/NVIDIA/NeMo-Retriever/blob/main/nemo_retriever/README.md)).
- Sample PDF at `../data/multimodal_test.pdf`.
- LanceDB and PyArrow available in your environment.

In [ ]:
# Optional: install LanceDB dependencies if needed
# %pip install -qU lancedb pyarrow

## VDB overview

`nemo_retriever` writes embedded chunks to LanceDB via `VdbUploadParams` during `GraphIngestor.ingest()`. Query the table with `nemo_retriever.retriever.Retriever` (same embedding model family as ingestion).

## Step 1: Ingest into LanceDB

Run extract, embed, and VDB upload in one `GraphIngestor` pipeline.

In [ ]:
from nemo_retriever.graph_ingestor import GraphIngestor
from nemo_retriever.params import EmbedParams, ExtractParams, VdbUploadParams

ingestor = (
    GraphIngestor(run_mode="inprocess", documents=["../data/multimodal_test.pdf"])
    .extract(
        ExtractParams(
            extract_text=True,
            extract_tables=True,
            extract_images=False,
        )
    )
    .embed(EmbedParams(model_name="nvidia/llama-nemotron-embed-1b-v2", local_ingest_embed_backend="hf"))
    .vdb_upload(
        VdbUploadParams(
            vdb_op="lancedb",
            uri="./lancedb",
            table_name="nv-ingest",
            overwrite=True,
        )
    )
)
ingestor.ingest()

The LanceDB database is created at `./lancedb` with table `nv-ingest`. Re-run with `overwrite=True` to recreate the table.

## Step 2: Search with Retriever

Embed the query locally and run vector search against the LanceDB table.

In [ ]:
from nemo_retriever.retriever import Retriever

retriever = Retriever(
    lancedb_uri="./lancedb",
    lancedb_table="nv-ingest",
    embedder="nvidia/llama-nemotron-embed-1b-v2",
    top_k=5,
    local_query_embed_backend="hf",
)

for question in [
    "What is shown in the charts?",
    "Summarize the table contents.",
]:
    hits = retriever.query(question)
    print(question, "->", len(hits), "hits")
    if hits:
        print(hits[0].get("text", "")[:200])